# Milestone 4: Embedding-Based SKU Matching

This notebook demonstrates visual search retrieval and threshold calibration on the clean, leakage-free dataset splits. We will load the gallery index, query it using validation crops, compute retrieval metrics, and analyze failure cases.

In [ ]:
import os
import sys
import pickle
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Add workspace root to Python path
workspace_root = Path(os.getcwd()).parent
sys.path.append(str(workspace_root))

from ml.retrieval.base import verify_split_disjointness
from ml.retrieval.numpy_index import NumpyCosineIndex
from ml.evaluation.metrics import (
    compute_recall_at_k,
    compute_mrr,
    compute_ndcg_at_k,
    calibrate_similarity_threshold
)

### 1. Load the Gallery Registry & Query Embeddings

We load the serialized DINOv2 train split gallery and validation queries.

In [ ]:
gallery_path = workspace_root / "data/processed/crops/gt_clean/embeddings_dinov2_train.pkl"
query_path = workspace_root / "data/processed/crops/gt_clean/embeddings_dinov2_val.pkl"

print("Loading Gallery...")
with open(gallery_path, "rb") as f:
    gallery_db = pickle.load(f)

print("Loading Queries...")
with open(query_path, "rb") as f:
    query_db = pickle.load(f)

gallery_vectors = gallery_db["embeddings"]
gallery_records = gallery_db["crop_records"]

query_vectors = query_db["embeddings"]
query_records = query_db["crop_records"]

print(f"Gallery vectors shape: {gallery_vectors.shape}")
print(f"Query vectors shape: {query_vectors.shape}")


### 2. Verify Split Disjointness

Ensure that there is zero cross-split duplicate family leakage.

In [ ]:
try:
    verify_split_disjointness(gallery_records, query_records)
    print("SUCCESS: Splits are disjoint!")
except AssertionError as e:
    print(f"FAIL: Split leakage detected: {e}")

### 3. Build Cosine Index & Retrieve Nearest Neighbors

In [ ]:
index = NumpyCosineIndex(dimension=gallery_db["metadata"]["embedding_dimension"])
index.add(gallery_vectors, gallery_records)

neighbor_indices, similarity_scores = index.search(query_vectors, top_k=5)

neighbor_labels = np.array([
    [gallery_records[idx]["remapped_class_id"] for idx in row]
    for row in neighbor_indices
])
query_labels = np.array([r["remapped_class_id"] for r in query_records])

print("Top-5 neighbors similarity search complete.")

### 4. Evaluate Global Performance Statistics

In [ ]:
rec1 = compute_recall_at_k(neighbor_labels, query_labels, 1)
rec5 = compute_recall_at_k(neighbor_labels, query_labels, 5)
ndcg5 = compute_ndcg_at_k(neighbor_labels, query_labels, 5)
mrr = compute_mrr(neighbor_labels, query_labels)

print(f"Recall@1: {rec1 * 100:.2f}%")
print(f"Recall@5: {rec5 * 100:.2f}%")
print(f"NDCG@5:   {ndcg5 * 100:.2f}%")
print(f"MRR:       {mrr * 100:.2f}%")

### 5. Calibrate Similarity Thresholds & Sweeps

We sweep similarity thresholds to study the Precision and Automation Coverage trade-offs.

In [ ]:
top_scores = similarity_scores[:, 0]
top_correct = (neighbor_labels[:, 0] == query_labels)

thresholds = np.linspace(0.5, 1.0, 101)
precisions = []
automation_rates = []

for tau in thresholds:
    mask = (top_scores >= tau)
    num_auto = np.sum(mask)
    prec = np.sum(top_correct[mask]) / num_auto if num_auto > 0 else 1.0
    precisions.append(prec)
    automation_rates.append(num_auto / len(top_scores))

fig, ax1 = plt.subplots(figsize=(10, 6))
ax2 = ax1.twinx()

ax1.plot(thresholds, precisions, 'g-', label='Precision')
ax2.plot(thresholds, automation_rates, 'b-', label='Automation Rate')

ax1.set_xlabel('Similarity Threshold')
ax1.set_ylabel('Precision', color='g')
ax2.set_ylabel('Automation Rate', color='b')

calib_tau, calib_auto = calibrate_similarity_threshold(top_scores, top_correct, target_precision=0.95)
ax1.axvline(calib_tau, color='r', linestyle='--', label=f'Calibrated tau* = {calib_tau:.3f}')

plt.title('Precision & Automation Rate Sweep')
plt.grid(True)
plt.show()

print(f"Calibrated Threshold: {calib_tau:.3f} achieves {calib_auto * 100:.2f}% Automation Rate at 95% Precision.")